# 19b. AHP có ràng buộc cứng, CPU (GĐ J, bổ sung task 7.2)
Cùng ma trận đã duyệt `docs/ahp_matrix.json` và cùng số liệu `reports/tables/summary_test.csv` (số test đã có của GĐ H, **không chạy lại test**).
Kịch bản "sàng lọc rồi xếp hạng": loại phương án có > 100.000 tham số hoặc > 1 MB trước khi chuẩn hóa và xếp hạng.
Ghi `ahp_constrained.json` và `gd7_ahp_constrained.png`.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "không rõ"


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
INFO["cpu"] = cpu_model()
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| CPU:", INFO["cpu"])

## pytest (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]
    r2 = run([sys.executable, "-m", "pytest", "-q", "tests/test_ahp.py"], cwd=REPO)
    INFO["pytest_ahp"] = r2.stdout.strip().splitlines()[-1]
save_info()

## AHP có ràng buộc

In [ ]:
with step("ahp_constrained"):
    run([sys.executable, "-I", "scripts/ahp_constrained.py", "reports/tables/summary_test.csv", "docs/ahp_matrix.json", f"{WORK}/ahp_constrained.json", WORK], cwd=REPO)
res = json.load(open(f"{WORK}/ahp_constrained.json", encoding="utf-8"))
INFO["scenario_a_top"], INFO["scenario_b_top"] = res["scenario_a_unconstrained"]["top"], res["scenario_b_constrained"]["top"]
save_info()
print(json.dumps({"a": res["scenario_a_unconstrained"]["ranking"], "b": res["scenario_b_constrained"]["ranking"],
                  "excluded": res["scenario_b_constrained"]["excluded"]}, ensure_ascii=False, indent=1))
print(os.listdir(WORK))